# Dilated convolution kernel animation

This notebook makes a short video with three traces:

- **blue**: the input signal (fixed)
- **green**: the kernel (its weights and its dilation change over time)
- **orange**: the convolution output for the current kernel

The animation has two parts:

1. The kernel weights morph through `NUM_KERNELS` random kernels, using the first value of `DILATIONS`.
2. The last kernel is kept, and its dilation moves through the remaining values of `DILATIONS`.

Requires `numpy`, `matplotlib`, and `ffmpeg` on the PATH.

## 1. Settings
Change these values, then run all cells.

In [16]:
SIGNAL_FILE = "signal.csv"     # file with one signal value per line
NUM_KERNELS = 7                # how many random kernels to morph through
KERNEL_SIZE = 9                # number of weights in each kernel
DILATIONS   = [2, 8, 1, 8,16,4, 2]  # dilation values to visit, in order

HOLD   = 0.3                   # seconds to pause on each kernel / dilation
MORPH  = 0.6                   # seconds to move from one to the next
FPS    = 30                    # frames per second
SEED   = 7                     # random seed for the kernels
OUTPUT = "dilated_kernel.mp4"  # output video file

## 2. Imports and signal

In [17]:
import subprocess

import numpy as np
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure
from IPython.display import Video

signal = np.loadtxt(SIGNAL_FILE).ravel()
# Or paste the values directly instead of reading a file:
# signal = np.array([1.798, 9.751, 5.494, ...])

print(f"Signal: {len(signal)} samples")

Signal: 256 samples


## 3. Kernels and convolution

In [18]:
def make_random_kernels(num_kernels, kernel_size, seed):
    """Random kernels with zero mean, scaled so the largest weight is +/-1."""
    rng = np.random.default_rng(seed)
    kernels = rng.normal(size=(num_kernels, kernel_size))
    kernels -= kernels.mean(axis=1, keepdims=True)
    kernels /= np.abs(kernels).max(axis=1, keepdims=True)
    return kernels


def kernel_offsets(kernel_size, dilation):
    """Sample positions of the kernel taps, centred on zero."""
    return (np.arange(kernel_size) - (kernel_size - 1) / 2) * dilation


def dilated_convolution(signal, weights, dilation):
    """Convolve `signal` with `weights` at the given dilation (zero padded).

    The dilation may be fractional: the signal is linearly interpolated,
    which lets the animation move smoothly between integer dilations.
    """
    positions = np.arange(len(signal))
    taps = positions[:, None] + kernel_offsets(len(weights), dilation)[None, :]
    samples = np.interp(taps, positions, signal, left=0.0, right=0.0)
    return samples @ weights

## 4. Timeline
Which kernel and dilation to show at each moment.

In [19]:
def build_keyframes(kernels, dilations):
    """List of (weights, dilation) states the animation passes through."""
    first_dilation, last_kernel = dilations[0], kernels[-1]
    weight_part = [(k, first_dilation) for k in kernels]
    dilation_part = [(last_kernel, d) for d in dilations[1:]]
    return weight_part + dilation_part


def smoothstep(u):
    """Ease in and out: 0 -> 1 with zero speed at both ends."""
    u = np.clip(u, 0.0, 1.0)
    return u * u * (3 - 2 * u)


def state_at(time, keyframes, hold, morph):
    """Kernel weights and dilation at `time` seconds."""
    step = hold + morph                               # one hold + one transition
    i = min(int(time // step), len(keyframes) - 1)    # current keyframe
    if i == len(keyframes) - 1:
        return keyframes[-1]

    (w0, d0), (w1, d1) = keyframes[i], keyframes[i + 1]
    u = smoothstep((time - i * step - hold) / morph)  # 0 while holding
    weights = (1 - u) * w0 + u * w1
    dilation = np.exp((1 - u) * np.log(d0) + u * np.log(d1))  # geometric blend
    return weights, dilation

## 5. Drawing

In [20]:
WIDTH, HEIGHT = 1200, 800          # pixels
BLUE, ORANGE, GREEN, GREY = "#1f77b4", "#ff7f0e", "#2ca02c", "#bbbbbb"
LINE_WIDTH = 3

INPUT_BASELINE = 0.25              # vertical position of each trace
OUTPUT_BASELINE = -1.45
KERNEL_BASELINE = 1.10
INPUT_HEIGHT = 0.85                # peak height of each trace
OUTPUT_HEIGHT = 0.80
KERNEL_HEIGHT = 0.22


def setup_figure(signal):
    """Draw everything that never changes; return the two animated lines."""
    n = len(signal)
    fig = Figure(figsize=(WIDTH / 100, HEIGHT / 100), dpi=100)
    FigureCanvasAgg(fig)           # off-screen canvas, works with any backend
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis("off")
    ax.set_xlim(-0.04 * n, 1.04 * n)
    ax.set_ylim(-3.3, 2.1)

    # baselines
    ax.axhline(INPUT_BASELINE, color=GREY, ls="--", lw=LINE_WIDTH)
    ax.axhline(OUTPUT_BASELINE, color=GREY, ls="--", lw=LINE_WIDTH)
    ax.plot([0.5 * n, 1.02 * n], [KERNEL_BASELINE] * 2,
            color=GREEN, ls=":", lw=LINE_WIDTH, alpha=0.3)

    # input signal
    scaled = signal / np.abs(signal).max() * INPUT_HEIGHT
    ax.plot(np.arange(n), INPUT_BASELINE + scaled, color=BLUE, lw=LINE_WIDTH)

    # labels
    label = dict(fontsize=20, style="italic", ha="right", va="top")
    ax.text(n - 2, INPUT_BASELINE - 0.05, "input", color="#666666", **label)
    ax.text(n - 2, OUTPUT_BASELINE - 0.05, "output", color="#666666", **label)
    ax.text(n - 2, KERNEL_BASELINE - 0.05, "kernel", color=GREEN, **label)

    output_line, = ax.plot([], [], color=ORANGE, lw=LINE_WIDTH)
    kernel_line, = ax.plot([], [], color=GREEN, lw=LINE_WIDTH)
    return fig, output_line, kernel_line


def update_frame(signal, weights, dilation, output_line, kernel_line):
    """Move the kernel and output lines to the current state."""
    n = len(signal)

    output = dilated_convolution(signal, weights, dilation)
    peak = np.abs(output).max()
    if peak > 0:
        output = output / peak * OUTPUT_HEIGHT        # same height every frame
    output_line.set_data(np.arange(n), OUTPUT_BASELINE + output)

    kernel_x = 0.72 * n + kernel_offsets(len(weights), dilation)
    kernel_line.set_data(kernel_x, KERNEL_BASELINE + weights * KERNEL_HEIGHT)

## 6. Render the video

In [21]:
kernels = make_random_kernels(NUM_KERNELS, KERNEL_SIZE, SEED)
keyframes = build_keyframes(kernels, DILATIONS)

duration = (len(keyframes) - 1) * (HOLD + MORPH) + HOLD
num_frames = max(1, round(duration * FPS))

fig, output_line, kernel_line = setup_figure(signal)

# Send raw frames straight to ffmpeg, which encodes the video.
import os
import shutil

candidates = [
    shutil.which("ffmpeg"),
    "/opt/homebrew/bin/ffmpeg",
    os.path.expanduser("~/.local/bin/ffmpeg"),
]
ffmpeg_bin = next(
    (path for path in candidates if path and os.access(path, os.X_OK)),
    None,
)
if ffmpeg_bin is None:
    raise RuntimeError("ffmpeg was not found. Install it, then run this cell again.")

ffmpeg = subprocess.Popen(
    [ffmpeg_bin, "-v", "error", "-y",
     "-f", "rawvideo", "-pix_fmt", "rgba",
     "-s", f"{WIDTH}x{HEIGHT}", "-r", str(FPS), "-i", "-",
     "-c:v", "libx264", "-crf", "18", "-pix_fmt", "yuv420p", OUTPUT],
    stdin=subprocess.PIPE,
)
for frame in range(num_frames):
    weights, dilation = state_at(frame / FPS, keyframes, HOLD, MORPH)
    update_frame(signal, weights, dilation, output_line, kernel_line)
    fig.canvas.draw()
    ffmpeg.stdin.write(fig.canvas.buffer_rgba())
ffmpeg.stdin.close()

if ffmpeg.wait() != 0:
    raise RuntimeError("ffmpeg failed")
print(f"Wrote {OUTPUT}: {num_frames} frames, {duration:.1f} s")

Python(24256) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Wrote dilated_kernel.mp4: 333 frames, 11.1 s


## 7. Watch it

In [22]:
Video(OUTPUT, embed=True, width=720)